In [ ]:
import os, copy, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
from google.colab import files
uploaded = files.upload()
import zipfile
with zipfile.ZipFile("WaRP-C-preprocessed.zip", "r") as z:
    z.extractall("/content/WaRP-C-preprocessed")

Saving WaRP-C-preprocessed.zip to WaRP-C-preprocessed.zip


In [ ]:
PREPROCESSED_ROOT = "/content/WaRP-C-preprocessed"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CFG = {
    "num_classes":          28,
    "backbone_name":        "dinov2_vitb14",
    "freeze_backbone":      False,
    "lr":                   3e-4,
    "min_lr":               1e-6,
    "weight_decay":         0.05,
    "label_smoothing":      0.1,
    "warmup_epochs":        3,
    "num_epochs":           15,
    "blend_alpha":          0.4,
    "use_class_weights_loss": True,
    "early_stop_patience":  5,
}

full_train_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

eval_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def _make_flat_dataset(root_dir, transform):
    samples, class_to_idx = [], {}
    for superclass in sorted(os.listdir(root_dir)):
        sp = os.path.join(root_dir, superclass)
        if not os.path.isdir(sp): continue
        for subclass in sorted(os.listdir(sp)):
            scp = os.path.join(sp, subclass)
            if not os.path.isdir(scp): continue
            if subclass not in class_to_idx:
                class_to_idx[subclass] = len(class_to_idx)
            for img_name in os.listdir(scp):
                if img_name.lower().endswith(".jpg"):
                    samples.append((os.path.join(scp, img_name), class_to_idx[subclass]))
    dataset = datasets.ImageFolder(root_dir, transform=transform)
    dataset.samples = dataset.imgs = samples
    dataset.targets = [s[1] for s in samples]
    dataset.classes = list(class_to_idx.keys())
    dataset.class_to_idx = class_to_idx
    return dataset


def get_dataloaders(root=PREPROCESSED_ROOT, batch_size=32, num_workers=2, seed=42):
    torch.manual_seed(seed)
    train_ds = _make_flat_dataset(f"{root}/train", transform=full_train_pipeline)
    val_ds   = _make_flat_dataset(f"{root}/val",   transform=eval_pipeline)
    test_ds  = _make_flat_dataset(f"{root}/test",  transform=eval_pipeline)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


def get_class_counts(dataset):
    counts = np.bincount(dataset.targets, minlength=len(dataset.classes))
    return counts.astype(np.float32)


train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders()
num_classes = len(train_loader.dataset.classes)
print(f"Classes: {num_classes} | Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Classes: 28 | Train: 7058 | Val: 1765 | Test: 1551


In [ ]:
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(embed_dim, num_classes)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)


backbone = torch.hub.load('facebookresearch/dinov2', CFG["backbone_name"])
embed_dim = backbone.embed_dim
model = DINOv2Classifier(backbone, embed_dim, num_classes, freeze_backbone=CFG["freeze_backbone"]).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable:,}")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 286MB/s]


Total params: 86,602,012 | Trainable: 86,602,012


In [ ]:
# Class-balanced CrossEntropyLoss (inverse-frequency weighting),
if CFG["use_class_weights_loss"]:
    samples_per_class = get_class_counts(train_ds)
    freq_inverse = 1.0 / (samples_per_class + 1e-6)
    loss_weights = torch.tensor(freq_inverse / freq_inverse.sum(), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=CFG["label_smoothing"])
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

if CFG["freeze_backbone"]:
    optimizer = optim.AdamW(model.head.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
else:
    # LLRD: backbone gets LR/10 (gentle fine-tuning), head gets full LR
    optimizer = optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CFG["lr"] / 10},
        {"params": model.head.parameters(), "lr": CFG["lr"]},
    ], weight_decay=CFG["weight_decay"])

def get_lr_scale(epoch):
    if epoch < CFG["warmup_epochs"]:
        return (epoch + 1) / CFG["warmup_epochs"]
    decay_progress = (epoch - CFG["warmup_epochs"]) / max(1, CFG["num_epochs"] - CFG["warmup_epochs"])
    return CFG["min_lr"] / CFG["lr"] + 0.5 * (1 - CFG["min_lr"] / CFG["lr"]) * (1 + np.cos(np.pi * decay_progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, get_lr_scale)

In [ ]:
def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        blend_ratio = np.random.beta(alpha, alpha)
    else:
        blend_ratio = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    blended_imgs = blend_ratio * x + (1 - blend_ratio) * x[index, :]
    labels_orig, labels_mixed = y, y[index]
    return blended_imgs, labels_orig, labels_mixed, blend_ratio


def mixup_criterion(criterion, pred, labels_orig, labels_mixed, blend_ratio):
    return blend_ratio * criterion(pred, labels_orig) + (1 - blend_ratio) * criterion(pred, labels_mixed)


def train_one_epoch(model, loader, optimizer, criterion, blend_alpha=0.0):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if blend_alpha > 0:
            imgs, labels_orig, labels_mixed, blend_ratio = mixup_data(imgs, labels, blend_alpha)

        optimizer.zero_grad()
        logits = model(imgs)

        if blend_alpha > 0:
            loss = mixup_criterion(criterion, logits, labels_orig, labels_mixed, blend_ratio)
        else:
            loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

best_val_acc, best_state, patience_counter = 0.0, None, 0

for epoch in range(CFG["num_epochs"]):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, blend_alpha=CFG["blend_alpha"])
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    epoch_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(epoch_lr)

    print(f"Epoch [{epoch+1:02d}/{CFG['num_epochs']}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | LR: {epoch_lr:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  New best val acc: {best_val_acc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= CFG["early_stop_patience"]:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val_acc:.4f}")

Epoch [01/15] Train Loss: 3.0123 Acc: 0.191 | Val Loss: 2.6808 Acc: 0.569 | LR: 2.00e-05
  New best val acc: 0.5688
Epoch [02/15] Train Loss: 2.6031 Acc: 0.273 | Val Loss: 2.5524 Acc: 0.678 | LR: 3.00e-05
  New best val acc: 0.6776
Epoch [03/15] Train Loss: 2.5220 Acc: 0.282 | Val Loss: 2.7040 Acc: 0.468 | LR: 3.00e-05
Epoch [04/15] Train Loss: 2.4314 Acc: 0.291 | Val Loss: 2.4426 Acc: 0.687 | LR: 2.95e-05
  New best val acc: 0.6873
Epoch [05/15] Train Loss: 2.3209 Acc: 0.344 | Val Loss: 2.4105 Acc: 0.682 | LR: 2.80e-05
Epoch [06/15] Train Loss: 2.2716 Acc: 0.362 | Val Loss: 2.3952 Acc: 0.685 | LR: 2.56e-05
Epoch [07/15] Train Loss: 2.2166 Acc: 0.332 | Val Loss: 2.2878 Acc: 0.759 | LR: 2.25e-05
  New best val acc: 0.7586
Epoch [08/15] Train Loss: 2.0623 Acc: 0.388 | Val Loss: 2.3420 Acc: 0.711 | LR: 1.89e-05
Epoch [09/15] Train Loss: 2.0543 Acc: 0.428 | Val Loss: 2.2037 Acc: 0.782 | LR: 1.50e-05
  New best val acc: 0.7819
Epoch [10/15] Train Loss: 1.9596 Acc: 0.426 | Val Loss: 2.1770 A

In [ ]:
#Final test evaluation
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="weighted", zero_division=0)

print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(test_labels, test_preds)

Accuracy : 0.8169
Precision: 0.8305
Recall   : 0.8169
F1-score : 0.8188


In [ ]:
with open("dinov2_history.json", "w") as f:
    json.dump(history, f, indent=2)

torch.save({
    "model_state": model.state_dict(),
    "cfg": CFG,
    "test_acc": float(test_acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
}, "dinov2_final.pth")

from google.colab import files
files.download("dinov2_history.json")
files.download("dinov2_final.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>